<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Segments and different effects

**[Segments and different effects](https://learning.nextia-ai.com/courses/experiments/m03/segments-and-different-effects/)** · Experiments and Causal Impact: Did Our Change Work? · Module 3, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** estimate the effect in each segment, test whether two segments really differ, and measure how often 20 segments find a false winner when nothing changed.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Welch's t-test and the interval, from [The result and its uncertainty](https://learning.nextia-ai.com/courses/experiments/m03/the-result-and-its-uncertainty/); A/A tests, from [Check the test before the result](https://learning.nextia-ai.com/courses/experiments/m03/check-the-test-before-the-result/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m03/segments-and-different-effects/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M03-L04-segments-and-different-effects/segments-and-different-effects-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 "scipy==1.15.3; python_version < '3.14'" "scipy==1.18.1; python_version >= '3.14'" statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "delivery/customers.csv.gz": "c22521f272d36b36fd26234d53d04aac1c8be347a5014ee3e2aeeffcc34a0c19",
    "delivery/truth.json": "6f088f29a112de123bb1da3bac0940a28c5adb706e9f24007686c373c14ab6a4"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The effect on mobile and on desktop

Omar asks whether free delivery works better on mobile. The helper `welch` gives the difference B − A, its 95% interval, the p-value and the standard error.

> **Predict.** Will both segments show a clear gain (an interval above zero)?

In [ ]:
d = pd.read_csv("data/delivery/customers.csv.gz")

def welch(df, col="revenue"):
    """Difference B - A, its 95% interval, the p-value and the standard error (Welch's t-test)."""
    a, b = df.loc[df.variant == "A", col], df.loc[df.variant == "B", col]
    t = stats.ttest_ind(b, a, equal_var=False)
    ci = t.confidence_interval()
    se = np.sqrt(a.var() / len(a) + b.var() / len(b))
    return {"difference": round(float(b.mean() - a.mean()), 3), "low": round(float(ci.low), 3),
            "high": round(float(ci.high), 3), "p": float(f"{t.pvalue:.3g}"), "se": round(float(se), 3)}

by_device = pd.DataFrame({device: welch(part) for device, part in d.groupby("device")}).T
by_device["true effect"] = [0.751, 0.660]  # the simulator's truth in each segment
by_device

> **Check.** You should see desktop `0.731`, from `-0.716` to `2.179` (p = 0.322), and mobile `1.223`, from `0.155` to `2.29` (p = 0.0248).

Mobile "wins" and desktop does not. But the simulator's true effects are almost the same: +0.75 on desktop and +0.66 on mobile.

## 2. Do the two segments really differ?

To compare two segments, test their difference. The two estimates come from different customers, so the standard error of the gap is √(SE₁² + SE₂²).

In [ ]:
gap = by_device.loc["mobile", "difference"] - by_device.loc["desktop", "difference"]
se_gap = np.sqrt(by_device.loc["mobile", "se"] ** 2 + by_device.loc["desktop", "se"] ** 2)
print(f"mobile - desktop: {gap:+.2f} dollars, 95% interval {gap - 1.96 * se_gap:+.2f} to {gap + 1.96 * se_gap:+.2f}")

> **Check.** You should see `mobile - desktop: +0.49 dollars, 95% interval -1.31 to +2.29`.

The interval of the gap contains zero, and it is wide. The data cannot say that mobile and desktop differ. The true gap is −0.09.

## 3. Twenty segments with no effect

Now cut a test into 20 segments: 10 provinces × 2 devices. To be sure that there is no effect, use A/A tests made from group A alone: split its customers in two at random. Count the segments with p < 0.05. Do it 200 times.

> **Predict.** In how many of the 200 runs will at least one of the 20 segments "win"?

In [ ]:
A = d[d.variant == "A"].reset_index(drop=True)
codes, names = pd.factorize(A.province + " " + A.device)
revenue = A.revenue.to_numpy()
rng = np.random.default_rng(0)
winners = []
for _ in range(200):
    half = rng.random(len(A)) < 0.5
    w = 0
    for k in range(len(names)):
        in_seg = codes == k
        if stats.ttest_ind(revenue[in_seg & half], revenue[in_seg & ~half], equal_var=False).pvalue < 0.05:
            w += 1
    winners.append(w)
winners = np.array(winners)
print(f"{len(names)} segments; at least one false winner in {(winners >= 1).mean():.0%} of the runs; "
      f"{winners.mean():.2f} false winners per run")
print(pd.Series(winners).value_counts().sort_index().to_dict())

> **Check.** You should see `20 segments; at least one false winner in 66% of the runs; 1.00 false winners per run`. With 20 independent segments, theory says 1 − 0.95²⁰ = 64%.

So a report that shows "the winning segment" usually shows luck. Plan the segments before the test, report all of them, and treat a new segment as an idea for the next test.

## 4. Your turn: new and returning customers

> **Your turn.** Estimate the effect for returning customers (`new_customer == 0`) and for new customers (`new_customer == 1`) with `welch`. Then compute the gap, returning minus new, and its 95% interval as in section 2. Put the lower end of the interval in `gap_low`, rounded to 2 decimals.

In [ ]:
gap_low = ...  # the lower end of the 95% interval of (returning - new), rounded to 2 decimals

In [ ]:
expect('gap_low', gap_low, -1.06, tolerance=1e-06)

## Next

You can read segments without being fooled by them. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m03/segments-and-different-effects/) has the charts and the knowledge checks that count toward your certificate. Next, we see what happens when a test is looked at every day and stopped at the first "win".

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Peeking](https://learning.nextia-ai.com/courses/experiments/m04/peeking/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m03/segments-and-different-effects/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.